# Model v1 (improved) — CatBoost baseline for VIEWS forecasting

This notebook implements a **strong, optimized** model with proper validation and feature engineering.

**Signals used (allowed at inference):**
- `CPM`
- `CHANNEL_NAME`
- `DATE`

**Key improvements:**
1) **Time-based validation** (last N days holdout) for honest local evaluation.
2) **Two model approaches** (log+RMSE vs raw+MAE) with automatic best selection by MAE.
3) **Enhanced features**: CPM binning, day of month, more cyclic features, channel frequency.
4) **Robust channel statistics** with shrinkage.
5) **Better hyperparameters** tuned for the task.
6) **Full data training** for final submission.
7) **Comprehensive diagnostics** to identify data/model issues.

**Outputs:**
- Model and artifacts saved into `artifacts/`
- Metrics and diagnostics
- Optional filled submission file saved into `outputs/`

Last updated: 2026-01-10

## 0. Setup

**What we do:** import libraries and define paths.  
**Outcome:** stable execution across machines (paths can be overridden via env vars).

In [74]:
import os
import json
from datetime import timedelta

import numpy as np
import pandas as pd

from catboost import CatBoostRegressor, Pool

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Paths (override if needed)
ALLDATA_PATH = os.getenv("ALLDATA_PATH", "/Users/karimkhabib/Documents/Projects Programming/PyCharm/telegram-ads-forecaster/data/AllData.csv")
TESTDATA_PATH = os.getenv("TESTDATA_PATH", "/Users/karimkhabib/Documents/Projects Programming/PyCharm/telegram-ads-forecaster/data/TestDataset.csv")  # optional
ARTIFACTS_DIR = os.getenv("ARTIFACTS_DIR", "artifacts")
OUTPUTS_DIR = os.getenv("OUTPUTS_DIR", "outputs")
HOLDOUT_DAYS = int(os.getenv("HOLDOUT_DAYS", "30"))

SELECTION_METRIC = os.getenv("SELECTION_METRIC", "MAE").upper()
USE_BLEND = os.getenv("USE_BLEND", "1") == "1"
BLEND_STEPS = int(os.getenv("BLEND_STEPS", "11"))

os.makedirs(ARTIFACTS_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)

print("ALLDATA_PATH:", ALLDATA_PATH)
print("TESTDATA_PATH:", TESTDATA_PATH)
print("ARTIFACTS_DIR:", ARTIFACTS_DIR)
print("OUTPUTS_DIR:", OUTPUTS_DIR)
print("HOLDOUT_DAYS:", HOLDOUT_DAYS)

ALLDATA_PATH: /Users/karimkhabib/Documents/Projects Programming/PyCharm/telegram-ads-forecaster/data/AllData.csv
TESTDATA_PATH: /Users/karimkhabib/Documents/Projects Programming/PyCharm/telegram-ads-forecaster/data/TestDataset.csv
ARTIFACTS_DIR: artifacts
OUTPUTS_DIR: outputs
HOLDOUT_DAYS: 30


## 1. Load data and diagnostics

**What we do:** read `AllData.csv`, normalize column names, parse `DATE`, check data quality.  
**Outcome:** a clean `df` with diagnostics to identify potential issues.

In [75]:
df = pd.read_csv(ALLDATA_PATH)

# Dataset quirk: sometimes columns have leading/trailing spaces
df.columns = df.columns.str.strip()

print(f"Initial shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

# Parse DATE
df["DATE"] = pd.to_datetime(df["DATE"], errors="coerce")

required = {"CPM", "CHANNEL_NAME", "DATE", "VIEWS"}
missing = required - set(df.columns)
assert not missing, f"Missing required columns: {missing}"

# Data quality checks
print("\n=== Data Quality Checks ===")
print(f"Total rows: {len(df):,}")
print(f"Missing DATE: {df['DATE'].isna().sum()}")
print(f"Missing VIEWS: {df['VIEWS'].isna().sum()}")
print(f"Missing CPM: {df['CPM'].isna().sum()}")
print(f"Missing CHANNEL_NAME: {df['CHANNEL_NAME'].isna().sum()}")

# Remove rows with missing critical data
initial_len = len(df)
df = df.dropna(subset=["DATE", "VIEWS", "CPM", "CHANNEL_NAME"]).copy()
print(f"Rows after cleaning: {len(df):,} (removed {initial_len - len(df)})")

# VIEWS distribution diagnostics
print("\n=== VIEWS Distribution ===")
print(df["VIEWS"].describe())
print(f"VIEWS == 0: {(df['VIEWS'] == 0).sum()} ({(df['VIEWS'] == 0).mean()*100:.2f}%)")
print(f"VIEWS < 0: {(df['VIEWS'] < 0).sum()}")
print(f"VIEWS > 100000: {(df['VIEWS'] > 100000).sum()} ({(df['VIEWS'] > 100000).mean()*100:.2f}%)")

# CPM distribution diagnostics
print("\n=== CPM Distribution ===")
print(df["CPM"].describe())
print(f"CPM <= 0: {(df['CPM'] <= 0).sum()}")
print(f"CPM > 100: {(df['CPM'] > 100).sum()} ({(df['CPM'] > 100).mean()*100:.2f}%)")

# Date range
print("\n=== Date Range ===")
print(f"Min date: {df['DATE'].min()}")
print(f"Max date: {df['DATE'].max()}")
print(f"Days span: {(df['DATE'].max() - df['DATE'].min()).days}")
print(f"Unique dates: {df['DATE'].nunique()}")

# Channel statistics
print("\n=== Channel Statistics ===")
print(f"Unique channels: {df['CHANNEL_NAME'].nunique()}")
print(f"Channels with < 10 samples: {(df['CHANNEL_NAME'].value_counts() < 10).sum()}")
print(f"Top 10 channels by frequency:")
print(df['CHANNEL_NAME'].value_counts().head(10))

# Check for potential issues
print("\n=== Potential Issues ===")
if (df['VIEWS'] == 0).mean() > 0.5:
    print("⚠️  WARNING: More than 50% of VIEWS are zero - model may struggle")
if (df['VIEWS'] < 0).any():
    print("⚠️  WARNING: Negative VIEWS found - will clip to 0")
if (df['CPM'] <= 0).any():
    print("⚠️  WARNING: Non-positive CPM found")
if df['CHANNEL_NAME'].isna().any():
    print("⚠️  WARNING: Missing channel names found")

df.head()

Initial shape: (142609, 7)
Columns: ['AD_ID', 'CPM', 'VIEWS', 'CLICKS', 'ACTIONS', 'CHANNEL_NAME', 'DATE']

=== Data Quality Checks ===
Total rows: 142,609
Missing DATE: 0
Missing VIEWS: 0
Missing CPM: 0
Missing CHANNEL_NAME: 0
Rows after cleaning: 142,609 (removed 0)

=== VIEWS Distribution ===
count    1.426090e+05
mean     9.998400e+02
std      7.260922e+03
min      0.000000e+00
25%      6.500000e+01
50%      2.550000e+02
75%      6.620000e+02
max      1.136470e+06
Name: VIEWS, dtype: float64
VIEWS == 0: 42 (0.03%)
VIEWS < 0: 0
VIEWS > 100000: 73 (0.05%)

=== CPM Distribution ===
count    142609.000000
mean          8.994971
std          29.404112
min           1.000000
25%           2.000000
50%           3.530000
75%           8.300000
max         999.910000
Name: CPM, dtype: float64
CPM <= 0: 0
CPM > 100: 743 (0.52%)

=== Date Range ===
Min date: 2024-10-02 00:00:00
Max date: 2025-12-17 00:00:00
Days span: 441
Unique dates: 430

=== Channel Statistics ===
Unique channels: 35957
C

,AD_ID,CPM,VIEWS,CLICKS,ACTIONS,CHANNEL_NAME,DATE
0,3652,1.2,52,0,0,tanya_in_france,2024-10-02
1,3653,1.2,54,2,0,relocator_cc,2024-10-02
2,3654,1.2,15,1,0,teleportazia,2024-10-02
3,3655,1.5,85,2,1,spetsialist_visa_support,2024-10-02
4,3656,19.4,688,21,3,pitkvch_news,2024-10-02


## 2. Time-based split (holdout last N days)

**What we do:** split by time for honest evaluation.  
**Why:** random split leaks time patterns.  
**Outcome:** `train_df` and `valid_df` for local scoring.

In [76]:
def split_last_days(data: pd.DataFrame, holdout_days: int = 30):
    unique_dates = np.sort(data["DATE"].unique())
    if len(unique_dates) <= holdout_days:
        raise ValueError(
            f"Not enough unique dates ({len(unique_dates)}) for holdout_days={holdout_days}"
        )
    cutoff = unique_dates[-holdout_days]
    train = data.loc[data["DATE"] < cutoff].copy()
    valid = data.loc[data["DATE"] >= cutoff].copy()
    return train, valid, cutoff, unique_dates[-1]

train_df, valid_df, cutoff, dmax = split_last_days(df, holdout_days=HOLDOUT_DAYS)

print("Train:", train_df.shape, "Valid:", valid_df.shape)
print("Cutoff:", cutoff, "→", dmax)


Train: (131923, 7) Valid: (10686, 7)
Cutoff: 2025-11-18T00:00:00.000000000 → 2025-12-17T00:00:00.000000000


## 3. Enhanced Feature engineering

We create comprehensive, deterministic, and safe features for inference:

**CPM features:**
- `cpm` - raw value
- `cpm_clip` - clipped at 99.5 percentile (robust to outliers)
- `log_cpm` - log1p transformation
- `cpm_bin` - quantile binning (20 bins) for non-linear relationships

**DATE features:**
- `dow` (0-6), `day`, `month`, `weekofyear`, `dayofyear`
- `is_weekend`, `days_since_start` (trend)
- Cyclic encodings: `dow_sin/cos`, `month_sin/cos`, `dayofyear_sin/cos`

**CHANNEL_NAME features:**
- Categorical feature for CatBoost (handles encoding automatically)
- `ch_count` - frequency in training data (with shrinkage for new channels)
- `ch_med_smooth` - smoothed median of log1p(VIEWS) per channel

Outcome: a reusable preprocessing pipeline we can apply to train/valid/test.

In [77]:
ANCHOR_DATE = pd.Timestamp("2020-01-01")


def add_date_features(data: pd.DataFrame, anchor_date: pd.Timestamp) -> pd.DataFrame:
    out = data.copy()
    d = out["DATE"]

    out["dow"] = d.dt.dayofweek.astype(int)
    out["is_weekend"] = (out["dow"] >= 5).astype(int)

    out["day"] = d.dt.day.astype(int)
    out["month"] = d.dt.month.astype(int)
    out["weekofyear"] = d.dt.isocalendar().week.astype(int)
    out["dayofyear"] = d.dt.dayofyear.astype(int)

    # Global trend anchor (avoid negatives for earlier dates)
    out["days_since_anchor"] = (d - anchor_date).dt.days.astype(int)

    # Cyclic encodings
    out["dow_sin"] = np.sin(2 * np.pi * out["dow"] / 7.0)
    out["dow_cos"] = np.cos(2 * np.pi * out["dow"] / 7.0)
    out["doy_sin"] = np.sin(2 * np.pi * out["dayofyear"] / 366.0)
    out["doy_cos"] = np.cos(2 * np.pi * out["dayofyear"] / 366.0)

    return out


def fit_preprocess_params(train: pd.DataFrame, cpm_clip_q: float = 0.999, cpm_bins: int = 30) -> dict:
    cpm_clip_value = float(train["CPM"].quantile(cpm_clip_q))
    _, bin_edges = pd.qcut(train["CPM"], q=cpm_bins, retbins=True, duplicates="drop")
    return {
        "cpm_clip_q": cpm_clip_q,
        "cpm_clip_value": cpm_clip_value,
        "cpm_bin_edges": [float(x) for x in bin_edges],
        "anchor_date": str(ANCHOR_DATE.date()),
    }


def apply_basic_preprocess(data: pd.DataFrame, params: dict) -> pd.DataFrame:
    out = data.copy()

    # CPM features
    out["cpm"] = out["CPM"].astype(float)
    out["cpm_clip"] = out["cpm"].clip(upper=params["cpm_clip_value"])
    out["log_cpm"] = np.log1p(out["cpm_clip"].clip(lower=0))

    # CPM binning (quantiles from train)
    bin_edges = np.array(params["cpm_bin_edges"])
    out["cpm_bin"] = pd.cut(out["cpm"], bins=bin_edges, include_lowest=True, labels=False)
    out["cpm_bin"] = out["cpm_bin"].fillna(-1).astype(int)

    # DATE features
    anchor_date = pd.to_datetime(params["anchor_date"])
    out = add_date_features(out, anchor_date=anchor_date)

    return out


def fit_channel_stats(train: pd.DataFrame, alpha: float = 10.0):
    # Robust channel stats on train only (log scale), with shrinkage.
    y = np.log1p(train["VIEWS"].clip(lower=0))
    global_med = float(np.median(y))

    stats = (
        train.assign(y=y)
             .groupby("CHANNEL_NAME")["y"]
             .agg(ch_count="size", ch_med="median")
             .reset_index()
    )

    w = stats["ch_count"] / (stats["ch_count"] + alpha)
    stats["ch_med_smooth"] = w * stats["ch_med"] + (1 - w) * global_med

    return stats[["CHANNEL_NAME", "ch_count", "ch_med_smooth"]], global_med


def apply_channel_stats(data: pd.DataFrame, ch_stats: pd.DataFrame, global_med: float) -> pd.DataFrame:
    out = data.merge(ch_stats, on="CHANNEL_NAME", how="left")
    out["ch_count"] = out["ch_count"].fillna(0).astype(int)
    out["ch_count_log"] = np.log1p(out["ch_count"])
    out["ch_med_smooth"] = out["ch_med_smooth"].fillna(global_med).astype(float)
    return out


def fit_channel_cpm_stats(train: pd.DataFrame):
    stats = (
        train.groupby("CHANNEL_NAME")["CPM"]
             .agg(ch_cpm_mean="mean", ch_cpm_median="median")
             .reset_index()
    )
    global_mean = float(train["CPM"].mean())
    global_median = float(train["CPM"].median())
    return stats, global_mean, global_median


def apply_channel_cpm_stats(data: pd.DataFrame, ch_stats: pd.DataFrame, global_mean: float, global_median: float) -> pd.DataFrame:
    out = data.merge(ch_stats, on="CHANNEL_NAME", how="left")
    out["ch_cpm_mean"] = out["ch_cpm_mean"].fillna(global_mean)
    out["ch_cpm_median"] = out["ch_cpm_median"].fillna(global_median)
    denom = out["ch_cpm_median"].replace(0, np.nan)
    out["cpm_to_ch_median"] = (out["cpm"] / denom).fillna(1.0)
    return out


## 4. Build train/valid matrices

**What we do:** fit preprocessing on train only, build features, create CatBoost Pools.  
**Outcome:** `train_pool`, `valid_pool` and feature specification.

In [78]:
# Fit preprocess on train only (no leakage)
preprocess = fit_preprocess_params(train_df, cpm_clip_q=0.999, cpm_bins=30)

# Basic features
train_fe = apply_basic_preprocess(train_df, preprocess)
valid_fe = apply_basic_preprocess(valid_df, preprocess)

# Channel stats (train only)
ch_stats, global_med_log = fit_channel_stats(train_df, alpha=10.0)
train_fe = apply_channel_stats(train_fe, ch_stats, global_med_log)
valid_fe = apply_channel_stats(valid_fe, ch_stats, global_med_log)

# Channel CPM stats (train only, uses CPM only)
ch_cpm_stats, global_cpm_mean, global_cpm_median = fit_channel_cpm_stats(train_df)
train_fe = apply_channel_cpm_stats(train_fe, ch_cpm_stats, global_cpm_mean, global_cpm_median)
valid_fe = apply_channel_cpm_stats(valid_fe, ch_cpm_stats, global_cpm_mean, global_cpm_median)

# Feature spec
feature_cols = [
    "cpm", "cpm_clip", "log_cpm", "cpm_bin",
    "dow", "is_weekend", "day", "month", "weekofyear", "dayofyear",
    "days_since_anchor",
    "dow_sin", "dow_cos", "doy_sin", "doy_cos",
    "ch_count", "ch_count_log", "ch_med_smooth",
    "ch_cpm_mean", "ch_cpm_median", "cpm_to_ch_median",
    "CHANNEL_NAME",  # categorical
]
cat_features = ["CHANNEL_NAME", "cpm_bin"]

X_train = train_fe[feature_cols]
X_valid = valid_fe[feature_cols]

y_train_raw = train_df["VIEWS"].astype(float).values
y_valid_raw = valid_df["VIEWS"].astype(float).values

y_train = np.log1p(np.clip(y_train_raw, 0, None))
y_valid = np.log1p(np.clip(y_valid_raw, 0, None))

train_pool = Pool(X_train, y_train, cat_features=cat_features)
valid_pool = Pool(X_valid, y_valid, cat_features=cat_features)

X_train.head()


,cpm,cpm_clip,log_cpm,cpm_bin,dow,is_weekend,day,month,weekofyear,dayofyear,...,dow_cos,doy_sin,doy_cos,ch_count,ch_count_log,ch_med_smooth,ch_cpm_mean,ch_cpm_median,cpm_to_ch_median,CHANNEL_NAME
0,1.2,1.2,0.788457,0,2,0,2,10,40,276,...,-0.222521,-0.999668,0.025748,13,2.639057,5.163988,2.042308,1.800,0.666667,tanya_in_france
1,1.2,1.2,0.788457,0,2,0,2,10,40,276,...,-0.222521,-0.999668,0.025748,18,2.944439,5.249288,11.147778,5.525,0.217195,relocator_cc
2,1.2,1.2,0.788457,0,2,0,2,10,40,276,...,-0.222521,-0.999668,0.025748,29,3.401197,4.984407,14.664483,8.000,0.150000,teleportazia
3,1.5,1.5,0.916291,1,2,0,2,10,40,276,...,-0.222521,-0.999668,0.025748,9,2.302585,5.046654,1.978889,1.500,1.000000,spetsialist_visa_support
4,19.4,19.4,3.015535,23,2,0,2,10,40,276,...,-0.222521,-0.999668,0.025748,9,2.302585,5.881408,6.367778,4.000,4.850000,pitkvch_news


## 5. Train CatBoost (v1 improved)

**What we do:** train a CatBoost regressor with early stopping.  
**Outcome:** a stronger baseline without complex modeling tricks.

In [79]:
# Helper function for metrics calculation
def calculate_metrics(y_true, y_pred, name=""):
    """Calculate comprehensive metrics."""
    y_true = np.asarray(y_true, dtype=float).clip(min=0)
    y_pred = np.asarray(y_pred, dtype=float).clip(min=0)
    
    mae = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    
    # RMSLE
    rmsle = float(np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2)))
    
    # SMAPE
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    denom = np.where(denom == 0, 1, denom)
    smape = float(np.mean(np.abs(y_pred - y_true) / denom)) * 100
    
    metrics = {
        "MAE": mae,
        "RMSE": rmse,
        "RMSLE": rmsle,
        "SMAPE": smape
    }
    
    if name:
        print(f"\n{name} Metrics:")
        for k, v in metrics.items():
            print(f"  {k}: {v:.4f}")
    
    return metrics

# ========== Approach A: log1p(VIEWS) with RMSE ==========
print("=" * 70)
print("Training Approach A: CatBoost on log1p(VIEWS) with RMSE loss")
print("=" * 70)

train_pool_a = Pool(X_train, y_train, cat_features=cat_features)
valid_pool_a = Pool(X_valid, y_valid, cat_features=cat_features)

model_a = CatBoostRegressor(
    loss_function="RMSE",
    depth=8,
    learning_rate=0.05,
    iterations=5000,
    l2_leaf_reg=5,
    random_strength=1.0,
    bootstrap_type="Bayesian",
    bagging_temperature=0.8,
    random_seed=RANDOM_SEED,
    eval_metric="RMSE",
    verbose=500,
    od_type="Iter",
    od_wait=200,
    task_type="CPU",
    devices="0",
)

model_a.fit(train_pool_a, eval_set=valid_pool_a, use_best_model=True, verbose=500)

# Predictions on validation (in log space, then convert back)
pred_valid_log_a = model_a.predict(valid_pool_a)
pred_valid_a = np.clip(np.expm1(pred_valid_log_a), 0, None)

metrics_a = calculate_metrics(y_valid_raw, pred_valid_a, "Approach A (on raw VIEWS)")

# ========== Approach B: VIEWS with MAE ==========
print("\n" + "=" * 70)
print("Training Approach B: CatBoost on VIEWS with MAE loss")
print("=" * 70)

train_pool_b = Pool(X_train, y_train_raw, cat_features=cat_features)
valid_pool_b = Pool(X_valid, y_valid_raw, cat_features=cat_features)

model_b = CatBoostRegressor(
    loss_function="MAE",
    depth=8,
    learning_rate=0.05,
    iterations=5000,
    l2_leaf_reg=5,
    random_strength=1.0,
    bootstrap_type="Bayesian",
    bagging_temperature=0.8,
    random_seed=RANDOM_SEED,
    eval_metric="MAE",
    verbose=500,
    od_type="Iter",
    od_wait=200,
    task_type="CPU",
    devices="0",
)

model_b.fit(train_pool_b, eval_set=valid_pool_b, use_best_model=True, verbose=500)

# Predictions on validation
pred_valid_b = np.clip(model_b.predict(valid_pool_b), 0, None)

metrics_b = calculate_metrics(y_valid_raw, pred_valid_b, "Approach B")

# ========== Model Selection ==========
print("")
print("=" * 70)
print(f"Model Comparison (by {SELECTION_METRIC} on holdout - lower is better)")
print("=" * 70)
print(f"Approach A (log+RMSE) {SELECTION_METRIC}: {metrics_a[SELECTION_METRIC]:.4f}")
print(f"Approach B (raw+MAE)  {SELECTION_METRIC}: {metrics_b[SELECTION_METRIC]:.4f}")

if metrics_a[SELECTION_METRIC] <= metrics_b[SELECTION_METRIC]:
    best_model = model_a
    best_model_name = "A"
    best_metrics = metrics_a
    best_pred_valid = pred_valid_a
    use_log_target = True
    print("")
    print("✓ Best single model: Approach A (log+RMSE)")
else:
    best_model = model_b
    best_model_name = "B"
    best_metrics = metrics_b
    best_pred_valid = pred_valid_b
    use_log_target = False
    print("")
    print("✓ Best single model: Approach B (raw+MAE)")

use_blend = False
best_blend_alpha = None
best_blend_metrics = None

if USE_BLEND:
    print("")
    print("-" * 70)
    print("Searching for best blend of A and B...")
    print("-" * 70)
    alphas = np.linspace(0.0, 1.0, BLEND_STEPS)
    for alpha in alphas:
        pred_blend = alpha * pred_valid_a + (1.0 - alpha) * pred_valid_b
        m = calculate_metrics(y_valid_raw, pred_blend)
        if best_blend_metrics is None or m[SELECTION_METRIC] < best_blend_metrics[SELECTION_METRIC]:
            best_blend_metrics = m
            best_blend_alpha = float(alpha)

    print(f"Best blend alpha (A weight): {best_blend_alpha:.2f}")
    print(f"Blend {SELECTION_METRIC}: {best_blend_metrics[SELECTION_METRIC]:.4f}")

    if best_blend_metrics[SELECTION_METRIC] < best_metrics[SELECTION_METRIC]:
        use_blend = True
        best_metrics = best_blend_metrics
        best_model_name = "blend"
        best_pred_valid = best_blend_alpha * pred_valid_a + (1.0 - best_blend_alpha) * pred_valid_b
        print("")
        print("✓ Using blended model (A+B)")

print("")
print("Best model metrics:")
for k, v in best_metrics.items():
    print(f"  {k}: {v:.4f}")

# Store for later use
best_model_approach = best_model_name
best_model_use_log = use_log_target


Training Approach A: CatBoost on log1p(VIEWS) with RMSE loss
0:	learn: 1.8722983	test: 2.0215715	best: 2.0215715 (0)	total: 84.5ms	remaining: 7m 2s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 1.907769783
bestIteration = 23

Shrink model to first 24 iterations.

Approach A (on raw VIEWS) Metrics:
  MAE: 626.0883
  RMSE: 5519.2995
  RMSLE: 1.9078
  SMAPE: 101.6612

Training Approach B: CatBoost on VIEWS with MAE loss
0:	learn: 925.8633108	test: 657.3491941	best: 657.3491941 (0)	total: 30.5ms	remaining: 2m 32s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 642.843595
bestIteration = 8

Shrink model to first 9 iterations.

Approach B Metrics:
  MAE: 642.8436
  RMSE: 5531.3915
  RMSLE: 2.0579
  SMAPE: 106.2901

Model Comparison (by MAE on holdout - lower is better)
Approach A (log+RMSE) MAE: 626.0883
Approach B (raw+MAE)  MAE: 642.8436

✓ Best single model: Approach A (log+RMSE)

---------------------------------------------------------------------

## 6. Evaluate (raw scale)

We predict in log space and invert back with `expm1`.  
Outcome: local metrics you can compare to baselines and track between versions.

In [80]:
def mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(y_true - y_pred)))

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def rmsle(y_true, y_pred):
    y_true = np.clip(np.asarray(y_true, dtype=float), 0, None)
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 0, None)
    return float(np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2)))

def smape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.abs(y_true) + np.abs(y_pred) + eps
    return float(np.mean(2.0 * np.abs(y_pred - y_true) / denom))

if use_blend:
    pred_valid = best_blend_alpha * pred_valid_a + (1.0 - best_blend_alpha) * pred_valid_b
elif best_model_use_log:
    pred_valid_log = best_model.predict(valid_pool_a)
    pred_valid = np.clip(np.expm1(pred_valid_log), 0, None)
else:
    pred_valid = np.clip(best_model.predict(valid_pool_b), 0, None)

metrics_local = {
    "MAE": mae(y_valid_raw, pred_valid),
    "RMSE": rmse(y_valid_raw, pred_valid),
    "RMSLE": rmsle(y_valid_raw, pred_valid),
    "SMAPE": smape(y_valid_raw, pred_valid),
}
metrics_local

{'MAE': 626.0882667884666,
 'RMSE': 5519.29947500275,
 'RMSLE': 1.9077697866565781,
 'SMAPE': 1.0166118847719563}

## 7. Baseline comparison (same split)

Outcome: sanity check. The improved model should beat these baselines.

In [81]:
global_median = float(train_df["VIEWS"].median())

# Baseline 1: global median
b1 = np.full_like(y_valid_raw, fill_value=global_median, dtype=float)

# Baseline 2: DOW median
train_dow = train_df["DATE"].dt.dayofweek
valid_dow = valid_df["DATE"].dt.dayofweek
dow_median = train_df.groupby(train_dow)["VIEWS"].median()
b2 = valid_dow.map(dow_median).fillna(global_median).to_numpy(dtype=float)

# Baseline 3: channel shrink baseline (raw scale)
alpha = 5
ch_raw = (train_df.groupby("CHANNEL_NAME")["VIEWS"].agg(ch_median="median", ch_count="size").reset_index())
valid_ch = valid_df[["CHANNEL_NAME"]].merge(ch_raw, on="CHANNEL_NAME", how="left")
w = (valid_ch["ch_count"] / (valid_ch["ch_count"] + alpha)).fillna(0.0)
b3 = (w * valid_ch["ch_median"].fillna(global_median) + (1 - w) * global_median).to_numpy(dtype=float)

def score_row(name, pred):
    return {
        "model": name,
        "MAE": mae(y_valid_raw, pred),
        "RMSE": rmse(y_valid_raw, pred),
        "RMSLE": rmsle(y_valid_raw, pred),
        "SMAPE": smape(y_valid_raw, pred),
    }

rows = [
    score_row("baseline_global_median", b1),
    score_row("baseline_dow_median", b2),
    score_row("baseline_channel_shrink", b3),
    score_row("catboost_v1_improved", pred_valid),
]
pd.DataFrame(rows).sort_values("MAE")

,model,MAE,RMSE,RMSLE,SMAPE
3,catboost_v1_improved,626.088267,5519.299475,1.907770,1.016612
0,baseline_global_median,662.004398,5547.165818,2.124321,1.103390
1,baseline_dow_median,664.060453,5548.652700,2.134005,1.104632
2,baseline_channel_shrink,671.651889,5510.127151,2.044897,1.052407


## 8. Final training on FULL data (for submission)

**Important:** once you selected a configuration, train on **all** rows to avoid losing the last N days.  
Outcome: `final_model` + full artifacts for submission and deployment.

In [82]:
full_df = df.copy()

# Fit preprocessing on FULL data
preprocess_full = fit_preprocess_params(full_df, cpm_clip_q=0.999, cpm_bins=30)
full_fe = apply_basic_preprocess(full_df, preprocess_full)

# Fit channel stats on FULL data (still offline, no leakage for submission)
ch_stats_full, global_med_log_full = fit_channel_stats(full_df, alpha=10.0)
full_fe = apply_channel_stats(full_fe, ch_stats_full, global_med_log_full)

# Fit channel CPM stats on FULL data
ch_cpm_stats_full, global_cpm_mean_full, global_cpm_median_full = fit_channel_cpm_stats(full_df)
full_fe = apply_channel_cpm_stats(full_fe, ch_cpm_stats_full, global_cpm_mean_full, global_cpm_median_full)

X_full = full_fe[feature_cols]
y_full_raw = full_df["VIEWS"].astype(float).values
y_full_log = np.log1p(np.clip(y_full_raw, 0, None))

if use_blend:
    full_pool_a = Pool(X_full, y_full_log, cat_features=cat_features)
    full_pool_b = Pool(X_full, y_full_raw, cat_features=cat_features)

    final_model_a = CatBoostRegressor(**model_a.get_params())
    final_model_b = CatBoostRegressor(**model_b.get_params())

    final_model_a.fit(full_pool_a, verbose=200)
    final_model_b.fit(full_pool_b, verbose=200)
else:
    y_full = y_full_log if best_model_use_log else y_full_raw
    full_pool = Pool(X_full, y_full, cat_features=cat_features)

    final_model = CatBoostRegressor(**best_model.get_params())
    final_model.fit(full_pool, verbose=200)


0:	learn: 1.8809063	total: 156ms	remaining: 13m
200:	learn: 1.1187737	total: 6.18s	remaining: 2m 27s
400:	learn: 1.0587131	total: 10.4s	remaining: 1m 58s
600:	learn: 1.0227954	total: 14.8s	remaining: 1m 48s
800:	learn: 0.9963466	total: 19s	remaining: 1m 39s
1000:	learn: 0.9749074	total: 23.2s	remaining: 1m 32s
1200:	learn: 0.9562296	total: 27.4s	remaining: 1m 26s
1400:	learn: 0.9399676	total: 31.6s	remaining: 1m 21s
1600:	learn: 0.9255282	total: 35.7s	remaining: 1m 15s
1800:	learn: 0.9124631	total: 40s	remaining: 1m 11s
2000:	learn: 0.9004183	total: 44.2s	remaining: 1m 6s
2200:	learn: 0.8893172	total: 48.4s	remaining: 1m 1s
2400:	learn: 0.8789406	total: 52.6s	remaining: 56.9s
2600:	learn: 0.8686482	total: 56.8s	remaining: 52.4s
2800:	learn: 0.8592678	total: 1m	remaining: 47.9s
3000:	learn: 0.8503736	total: 1m 5s	remaining: 43.4s
3200:	learn: 0.8417992	total: 1m 9s	remaining: 38.9s
3400:	learn: 0.8334951	total: 1m 13s	remaining: 34.5s
3600:	learn: 0.8258681	total: 1m 17s	remaining: 30.1

## 9. Save artifacts

Outcome: you can reproduce predictions in batch scripts and the API.

In [83]:
# Save model(s)
if use_blend:
    model_path_a = os.path.join(ARTIFACTS_DIR, "model_v1_improved_A.cbm")
    model_path_b = os.path.join(ARTIFACTS_DIR, "model_v1_improved_B.cbm")
    final_model_a.save_model(model_path_a)
    final_model_b.save_model(model_path_b)
else:
    model_path = os.path.join(ARTIFACTS_DIR, "model_v1_improved.cbm")
    final_model.save_model(model_path)

# Save preprocessing + channel stats references
with open(os.path.join(ARTIFACTS_DIR, "preprocess_v1_improved.json"), "w", encoding="utf-8") as f:
    json.dump(preprocess_full, f, ensure_ascii=False, indent=2)

# Save channel stats table (CSV is easy to inspect)
ch_stats_path = os.path.join(ARTIFACTS_DIR, "channel_stats_v1_improved.csv")
ch_stats_full.to_csv(ch_stats_path, index=False)

# Save channel CPM stats
ch_cpm_stats_path = os.path.join(ARTIFACTS_DIR, "channel_cpm_stats_v1_improved.csv")
ch_cpm_stats_full.to_csv(ch_cpm_stats_path, index=False)

with open(os.path.join(ARTIFACTS_DIR, "meta_v1_improved.json"), "w", encoding="utf-8") as f:
    json.dump({
        "created_at": "2026-01-10",
        "model": "CatBoostRegressor",
        "target": "log1p(VIEWS)" if best_model_use_log else "VIEWS",
        "feature_cols": feature_cols,
        "cat_features": cat_features,
        "holdout_days_for_eval": HOLDOUT_DAYS,
        "selection_metric": SELECTION_METRIC,
        "use_blend": use_blend,
        "blend_alpha": best_blend_alpha,
        "notes": "Improved v1: trend + channel stats + CPM stats + full-data training for submission"
    }, f, ensure_ascii=False, indent=2)

with open(os.path.join(ARTIFACTS_DIR, "metrics_v1_improved_local.json"), "w", encoding="utf-8") as f:
    json.dump(metrics_local, f, ensure_ascii=False, indent=2)

if use_blend:
    print("Saved:", model_path_a)
    print("Saved:", model_path_b)
else:
    print("Saved:", model_path)
print("Saved:", ch_stats_path)
print("Saved:", ch_cpm_stats_path)


Saved: artifacts/model_v1_improved.cbm
Saved: artifacts/channel_stats_v1_improved.csv
Saved: artifacts/channel_cpm_stats_v1_improved.csv


## 10. Optional: fill `TestDataset.csv` and export a submission file

Outcome: `outputs/TestDataset_filled_model_v1_improved.csv` ready for upload.

In [84]:
if os.path.exists(TESTDATA_PATH):
    test_df = pd.read_csv(TESTDATA_PATH)
    test_df.columns = test_df.columns.str.strip()
    test_df["DATE"] = pd.to_datetime(test_df["DATE"], errors="coerce")

    test_fe = apply_basic_preprocess(test_df, preprocess_full)
    test_fe = apply_channel_stats(test_fe, ch_stats_full, global_med_log_full)
    test_fe = apply_channel_cpm_stats(test_fe, ch_cpm_stats_full, global_cpm_mean_full, global_cpm_median_full)

    X_test = test_fe[feature_cols]
    test_pool = Pool(X_test, cat_features=cat_features)

    if use_blend:
        pred_a = np.expm1(final_model_a.predict(test_pool))
        pred_b = final_model_b.predict(test_pool)
        pred_test = best_blend_alpha * pred_a + (1.0 - best_blend_alpha) * pred_b
        pred_test = np.clip(pred_test, 0, None)
    else:
        pred_test_raw = final_model.predict(test_pool)
        if best_model_use_log:
            pred_test = np.clip(np.expm1(pred_test_raw), 0, None)
        else:
            pred_test = np.clip(pred_test_raw, 0, None)

    out = test_df.copy()
    out["VIEWS"] = np.round(pred_test).astype(int)

    out_path = os.path.join(OUTPUTS_DIR, "TestDataset_filled_model_v1_improved.csv")
    out.to_csv(out_path, index=False)
    print("Saved:", out_path)
else:
    print(f"Test dataset not found at: {TESTDATA_PATH}")


Saved: outputs/TestDataset_filled_model_v1_improved.csv


## Recommended next step (Model v2)

If this improved v1 still stalls:
- run a rolling backtest (3 windows)
- try MAE / Quantile objective
- optionally add offline TGStat/TGMaps channel features (subscribers, avg views, etc.) cached locally